# 12.4 時間框的邊緣如何處理？

# 第 12 章：沿用相同介面加入聲音

前置：第10章模態展開與第7章loss mask。頻譜可獨立讀；聲音生成與log-mel都從PyTorch寫起，不依賴Whisper或torchaudio。FSDD爲額外真實語音pilot；本章默認合成音高／時長任務。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像用放大鏡逐段看聲音：時間框可重疊**

框長決定分析範圍；hop 決定多久移動一次。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/frames.svg")))

**先想一想：**window只改邊緣，會影響所有頻率嗎？

短框直接截斷會在邊緣出現跳變，引出額外頻譜泄漏。Hann window把邊緣逐漸壓小，代價是主峯變寬。

**把直覺寫成數學：**w[n]=0.5(1−cos(2πn/N))；PyTorch默認periodic Hann，和N−1版本略有差異。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.multimodal import tone

frame = tone()[:400]
window = torch.hann_window(400)
print("邊緣window", window[:3], window[-3:])
print("前後energy", frame.square().sum().item(), (frame * window).square().sum().item())

**如何讀結果：**window能降低突變，但不能把不同長度或sample rate自動變一致。

**只改一件事：**只換成全1 rectangular window。
